# Load X and y

In [1]:
import pandas as pd
import numpy as np
import os, json, joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

X = pd.read_csv("../data_processed/X_features.csv")
y = pd.read_csv("../data_processed/y_target.csv")["efficient_flag"]

print("Loaded X shape:", X.shape)
print("Loaded y shape:", y.shape)
display(X.head(3))
display(y.head(3))


Loaded X shape: (612229, 8)
Loaded y shape: (612229,)


,type,organization,province,hour,weekday,month,lat,lon
0,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ",กรุงเทพมหานคร,23,1,12,13.72812,100.65617
1,{สะพาน},เขตสาทร,กรุงเทพมหานคร,5,6,9,13.72060,100.52649
2,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ",กรุงเทพมหานคร,10,2,12,13.68158,100.65440


0    0
1    0
2    0
Name: efficient_flag, dtype: int64

# Train/test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)


X_train: (489783, 8) X_test: (122446, 8)
y_train: (489783,) y_test: (122446,)


# Define categorical & numeric columns

In [3]:
categorical_features = ["type", "organization", "province"]
numeric_features = ["hour", "weekday", "month", "lat", "lon"]


# Preprocessing pipeline

In [4]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# Helper to evaluate models

In [5]:
def eval_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    print("Accuracy:", round(acc, 3), "F1(efficient=1):", round(f1, 3))
    return acc, f1

# Baseline model – Logistic Regression

In [6]:
baseline_clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

baseline_clf.fit(X_train, y_train)
acc_base, f1_base = eval_model("Logistic Regression", baseline_clf, X_test, y_test)



=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.79      0.81      0.80     72398
           1       0.72      0.68      0.70     50048

    accuracy                           0.76    122446
   macro avg       0.75      0.75      0.75    122446
weighted avg       0.76      0.76      0.76    122446

Confusion matrix:
 [[58953 13445]
 [15978 34070]]
Accuracy: 0.76 F1(efficient=1): 0.698


# Train Random Forest

In [7]:
rf_clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_clf.fit(X_train, y_train)
acc_rf, f1_rf = eval_model("Random Forest (balanced)", rf_clf, X_test, y_test)



=== Random Forest (balanced) ===
              precision    recall  f1-score   support

           0       0.75      0.71      0.73     72398
           1       0.62      0.66      0.64     50048

    accuracy                           0.69    122446
   macro avg       0.69      0.69      0.69    122446
weighted avg       0.70      0.69      0.70    122446

Confusion matrix:
 [[51725 20673]
 [16798 33250]]
Accuracy: 0.694 F1(efficient=1): 0.64


# Choose the best model

In [8]:
if f1_rf > f1_base:
    best_name = "Random Forest (balanced)"
    best_model = rf_clf
    best_acc, best_f1 = acc_rf, f1_rf
else:
    best_name = "Logistic Regression"
    best_model = baseline_clf
    best_acc, best_f1 = acc_base, f1_base

print("\nBest model:", best_name)
print("Best accuracy:", round(best_acc, 3))
print("Best F1(efficient=1):", round(best_f1, 3))



Best model: Logistic Regression
Best accuracy: 0.76
Best F1(efficient=1): 0.698


# Save best model + metrics

In [9]:
os.makedirs("../ML/models", exist_ok=True)
os.makedirs("../ML/results", exist_ok=True)

joblib.dump(best_model, "../ML/models/best_efficiency_model.pkl")

metrics = {
    "best_model": best_name,
    "accuracy": float(best_acc),
    "f1_efficient": float(best_f1),
    "log_reg": {"accuracy": float(acc_base), "f1_efficient": float(f1_base)},
    "rf_balanced": {"accuracy": float(acc_rf), "f1_efficient": float(f1_rf)},
}

with open("../ML/results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved best model and metrics:", metrics)


Saved best model and metrics: {'best_model': 'Logistic Regression', 'accuracy': 0.7597063195204417, 'f1_efficient': 0.6984205077744636, 'log_reg': {'accuracy': 0.7597063195204417, 'f1_efficient': 0.6984205077744636}, 'rf_balanced': {'accuracy': 0.6939793868317462, 'f1_efficient': 0.6396014273210799}}


In [10]:
import pandas as pd
import joblib
import os

# 1) Load the filtered dataframe used in ML
df = pd.read_csv("../data_processed/df_ml_base.csv")
print("df_ml_base shape:", df.shape)

# 2) Build X in the same way as training
feature_cols = [
    "type",
    "organization",
    "province",
    "hour",
    "weekday",
    "month",
    "lat",
    "lon",
]
X = df[feature_cols].copy()

# 3) Load the trained model
model = joblib.load("../ML/models/best_efficiency_model.pkl")

# 4) Predict
y_pred = model.predict(X)
y_prob = model.predict_proba(X)[:, 1]  # probability efficient

print("len(y_pred):", len(y_pred))

# These should now match:
# len(y_pred) == len(df)

# 5) Attach predictions to df
df["pred_efficient_flag"] = y_pred
df["pred_efficient_prob"] = y_prob

# 6) Save for Power BI
os.makedirs("../data_processed", exist_ok=True)
out_path = "../data_processed/for_powerbi_with_predictions.csv"
df.to_csv(out_path, index=False)

print("Saved:", out_path, "with shape:", df.shape)


df_ml_base shape: (612229, 18)
len(y_pred): 612229
Saved: ../data_processed/for_powerbi_with_predictions.csv with shape: (612229, 20)
